# 00 Sudan demo data

Inspects the U.S. Census Bureau Sudan files from HDX and writes `data/processed/sudan_states.gpkg`.

Before running: download `Sudan.gdb.zip` and `Sudan_uscb_202101.xlsx` from https://data.humdata.org/dataset/sudan-subnational-boundaries-and-tabular-data into `data/raw/`, then unzip `Sudan.gdb.zip` there so you have `data/raw/Sudan.gdb/`.

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

# Notebook runs from notebooks/, so data sits one level up
RAW = Path("../data/raw")
OUT = Path("../data/processed")
GDB = next(RAW.glob("*.gdb"))
XLSX = next(RAW.glob("*.xlsx"))
print(GDB, XLSX)

In [ ]:
# List every layer in the geodatabase with its geometry type
gpd.list_layers(GDB)

Pick the state-level (admin 1) layer from the list above and set its name below.

In [ ]:
# Set this to the admin 1 layer name shown above
STATE_LAYER = "CHANGE_ME"

states_raw = gpd.read_file(GDB, layer=STATE_LAYER)
print(states_raw.crs, len(states_raw))
states_raw.drop(columns="geometry").head()

In [ ]:
# List the sheets in the population workbook
pd.ExcelFile(XLSX).sheet_names

In [ ]:
# Set this to the sheet with population by state, then check where the header row is
SHEET = "CHANGE_ME"

pop_raw = pd.read_excel(XLSX, sheet_name=SHEET, header=None)
pop_raw.head(15)

Set the header row, the join key in each table, and the population column, using what you saw above.

In [ ]:
# Fill these in from the outputs above
HEADER_ROW = 0
KEY_IN_STATES = "CHANGE_ME"
KEY_IN_POP = "CHANGE_ME"
POP_COLUMN = "CHANGE_ME"

pop = pd.read_excel(XLSX, sheet_name=SHEET, header=HEADER_ROW)
pop = pop[[KEY_IN_POP, POP_COLUMN]].rename(columns={KEY_IN_POP: "key", POP_COLUMN: "pop"})
pop["pop"] = pd.to_numeric(pop["pop"], errors="coerce")
pop.head()

In [ ]:
# Join, report anything that failed to match
states = states_raw.rename(columns={KEY_IN_STATES: "key"})
states = states.merge(pop, on="key", how="left", indicator=True)
print(states["_merge"].value_counts())
states[states["_merge"] != "both"][["key"]]

In [ ]:
# Area in km2 from an equal-area projection, then density
states["area_km2"] = states.to_crs("ESRI:102022").area / 1e6
states["density"] = states["pop"] / states["area_km2"]
states[["key", "pop", "area_km2", "density"]].sort_values("density")

In [ ]:
# Keep a small clean file in EPSG:4326, simplified to keep the demo light
keep = states[["key", "pop", "area_km2", "density", "geometry"]].to_crs("EPSG:4326")
keep["geometry"] = keep.geometry.simplify(0.005, preserve_topology=True)
OUT.mkdir(parents=True, exist_ok=True)
keep.to_file(OUT / "sudan_states.gpkg", driver="GPKG")
print((OUT / "sudan_states.gpkg").stat().st_size / 1e6, "MB")

In [ ]:
# Quick visual check with plain Lonboard
from lonboard import viz

viz(keep)